In [3]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

data_path = "../data/raw/"

In [4]:
orders      = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
customers   = pd.read_csv(data_path + "olist_customers_dataset.csv")
payments    = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
products    = pd.read_csv(data_path + "olist_products_dataset.csv")

print("All datasets loaded.")

All datasets loaded.


In [6]:
#Aggregate order_items per order
items_agg = order_items.groupby('order_id').agg(
    num_items       = ('order_item_id', 'count'),
    total_price     = ('price', 'sum'),
    total_freight   = ('freight_value', 'sum')
).reset_index()

In [7]:
#Aggregate Payments per order
payments_agg = payments.groupby('order_id').agg(
    payment_type        = ('payment_type', lambda x: x.mode()[0]),
    payment_installments= ('payment_installments', 'max'),
    payment_value       = ('payment_value', 'sum')
).reset_index()

In [9]:
#Merge everything into one dataframe
df = orders.merge(items_agg,    on='order_id', how='left')
df = df.merge(payments_agg,     on='order_id', how='left')
df = df.merge(customers[['customer_id','customer_state']], on='customer_id', how='left')

print("Master table shape:", df.shape)
df.head()

Master table shape: (99441, 15)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,num_items,total_price,total_freight,payment_type,payment_installments,payment_value,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,29.99,8.72,voucher,1.0,38.71,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,118.70,22.76,boleto,1.0,141.46,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,159.90,19.22,credit_card,3.0,179.12,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,45.00,27.20,credit_card,1.0,72.20,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,19.90,8.72,credit_card,1.0,28.62,SP


In [11]:
#create target variable
df['cancelled'] = (df['order_status'] == 'canceled').astype(int)

print("Cancellation rate:")
print(df['cancelled'].value_counts())
print(f"\n{df['cancelled'].mean()*100:.2f}% of orders were cancelled")

Cancellation rate:
cancelled
0    98816
1      625
Name: count, dtype: int64

0.63% of orders were cancelled


In [12]:
#datetime features
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

df['purchase_hour']       = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek']  = df['order_purchase_timestamp'].dt.dayofweek  # 0=Mon, 6=Sun
df['purchase_month']      = df['order_purchase_timestamp'].dt.month
df['is_weekend']          = df['purchase_dayofweek'].isin([5, 6]).astype(int)

print("Datetime features created.")

Datetime features created.


In [13]:
#Finalize columns to use
features = [
    'num_items', 'total_price', 'total_freight',
    'payment_type', 'payment_installments', 'payment_value',
    'customer_state', 'purchase_hour', 'purchase_dayofweek',
    'purchase_month', 'is_weekend', 'cancelled'
]

df_model = df[features].copy()
print("Shape:", df_model.shape)
df_model.isnull().sum()

Shape: (99441, 12)


num_items               775
total_price             775
total_freight           775
payment_type              1
payment_installments      1
payment_value             1
customer_state            0
purchase_hour             0
purchase_dayofweek        0
purchase_month            0
is_weekend                0
cancelled                 0
dtype: int64

In [14]:
#missing values 
# Numeric — fill with median
num_cols = ['num_items','total_price','total_freight',
            'payment_installments','payment_value']

for col in num_cols:
    df_model[col].fillna(df_model[col].median(), inplace=True)

# Categorical — fill with 'unknown'
df_model['payment_type'].fillna('unknown', inplace=True)
df_model['customer_state'].fillna('unknown', inplace=True)

print("Missing values after fill:")
print(df_model.isnull().sum())

Missing values after fill:
num_items               775
total_price             775
total_freight           775
payment_type              1
payment_installments      1
payment_value             1
customer_state            0
purchase_hour             0
purchase_dayofweek        0
purchase_month            0
is_weekend                0
cancelled                 0
dtype: int64


In [15]:
#encoding 
df_model = pd.get_dummies(df_model, 
                           columns=['payment_type', 'customer_state'], 
                           drop_first=True)

print("Shape after encoding:", df_model.shape)
df_model.head()

Shape after encoding: (99441, 40)


,num_items,total_price,total_freight,payment_installments,payment_value,purchase_hour,purchase_dayofweek,purchase_month,is_weekend,cancelled,...,customer_state_PR,customer_state_RJ,customer_state_RN,customer_state_RO,customer_state_RR,customer_state_RS,customer_state_SC,customer_state_SE,customer_state_SP,customer_state_TO
0,1.0,29.99,8.72,1.0,38.71,10,0,10,0,0,...,False,False,False,False,False,False,False,False,True,False
1,1.0,118.70,22.76,1.0,141.46,20,1,7,0,0,...,False,False,False,False,False,False,False,False,False,False
2,1.0,159.90,19.22,3.0,179.12,8,2,8,0,0,...,False,False,False,False,False,False,False,False,False,False
3,1.0,45.00,27.20,1.0,72.20,19,5,11,1,0,...,False,False,True,False,False,False,False,False,False,False
4,1.0,19.90,8.72,1.0,28.62,21,1,2,0,0,...,False,False,False,False,False,False,False,False,True,False


In [16]:
#Train/Test:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_model.drop('cancelled', axis=1)
y = df_model['cancelled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

Train size: (79552, 39)
Test size:  (19889, 39)


In [17]:
#Scaling 
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done.")

Scaling done.


In [ ]:
import pickle

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save arrays
np.save("../data/processed/X_train.npy", X_train_scaled)
np.save("../data/processed/X_test.npy",  X_test_scaled)
np.save("../data/processed/y_train.npy", y_train.values)
np.save("../data/processed/y_test.npy",  y_test.values)

# Save scaler and column names for later use
with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

pd.Series(X_train.columns).to_csv("../data/processed/feature_columns.csv", index=False)

print("All processed data saved.")